# CLEAN

## IMPORTS

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import os

##  DataLoader Class

In [ ]:
class DiabeticDataLoader:
    def __init__(self, df_source):
        self.df_source = df_source.copy()
        self.constant_cols = [
            col_name
            for col_name in self.df_source.columns
            if self.df_source[col_name].nunique(dropna=False) == 1
        ]
        self.identifier_cols = ["encounter_id", "patient_nbr"]
        self.target_col = "readmitted"
        self.df_clean = self._build_clean_dataframe()

    def _build_clean_dataframe(self):
        df_cleaned = self.df_source.drop(
            columns=self.constant_cols, errors="ignore"
        ).copy()

        if "max_glu_serum" in df_cleaned.columns:
            df_cleaned["max_glu_serum"] = df_cleaned["max_glu_serum"].replace(
                "None", np.nan
            )
            df_cleaned["max_glu_serum_measured"] = (
                df_cleaned["max_glu_serum"].notna().astype(int)
            )

        if "A1Cresult" in df_cleaned.columns:
            df_cleaned["A1Cresult"] = df_cleaned["A1Cresult"].replace("None", np.nan)
            df_cleaned["A1Cresult_measured"] = (
                df_cleaned["A1Cresult"].notna().astype(int)
            )

        if "weight" in df_cleaned.columns:
            df_cleaned["weight"] = df_cleaned["weight"].replace("?", np.nan)
            df_cleaned["weight_recorded"] = df_cleaned["weight"].notna().astype(int)
            df_cleaned["weight_bin"] = df_cleaned["weight"].astype("category")

        categorical_cols = [
            "race",
            "gender",
            "age",
            "weight",
            "payer_code",
            "medical_specialty",
            "diag_1",
            "diag_2",
            "diag_3",
            "max_glu_serum",
            "A1Cresult",
            "metformin",
            "repaglinide",
            "nateglinide",
            "chlorpropamide",
            "glimepiride",
            "acetohexamide",
            "glipizide",
            "glyburide",
            "tolbutamide",
            "pioglitazone",
            "rosiglitazone",
            "acarbose",
            "miglitol",
            "troglitazone",
            "tolazamide",
            "insulin",
            "glyburide_metformin",
            "glipizide_metformin",
            "glimepiride_pioglitazone",
            "metformin_rosiglitazone",
            "metformin_pioglitazone",
            "change",
            "diabetesMed",
            "readmitted",
            "weight_bin",
        ]
        for col_name in categorical_cols:
            if col_name in df_cleaned.columns:
                df_cleaned[col_name] = df_cleaned[col_name].astype("category")

        numeric_cols = [
            "encounter_id",
            "patient_nbr",
            "admission_type_id",
            "discharge_disposition_id",
            "admission_source_id",
            "time_in_hospital",
            "num_lab_procedures",
            "num_procedures",
            "num_medications",
            "number_outpatient",
            "number_emergency",
            "number_inpatient",
            "number_diagnoses",
            "max_glu_serum_measured",
            "A1Cresult_measured",
            "weight_recorded",
        ]
        for col_name in numeric_cols:
            if col_name in df_cleaned.columns:
                df_cleaned[col_name] = pd.to_numeric(
                    df_cleaned[col_name], errors="coerce"
                )

        return df_cleaned.reset_index(drop=True)

    def get_clean_data(self):
        return self.df_clean.copy()

    def get_features_target(self, target_col=None, drop_identifier_cols=True):
        target_name = target_col if target_col is not None else self.target_col
        df_model = self.df_clean.copy()

        if drop_identifier_cols:
            cols_to_drop = [
                col for col in self.identifier_cols if col in df_model.columns
            ]
        else:
            cols_to_drop = []

        if target_name not in df_model.columns:
            raise KeyError(
                f"Target column '{target_name}' not found in cleaned dataframe."
            )

        df_features = df_model.drop(
            columns=cols_to_drop + [target_name], errors="ignore"
        )
        df_target = df_model[target_name].copy()
        return df_features, df_target

    def get_train_test_split(
        self, target_col=None, test_size=0.2, random_state=42, stratify=True
    ):
        df_features, df_target = self.get_features_target(
            target_col=target_col, drop_identifier_cols=True
        )
        df_features_encoded = pd.get_dummies(df_features, drop_first=True)
        df_features_encoded = df_features_encoded.astype(float)

        stratify_target = df_target if stratify else None
        if stratify and df_target.nunique(dropna=False) < 2:
            stratify_target = None

        X_train, X_test, y_train, y_test = train_test_split(
            df_features_encoded,
            df_target,
            test_size=test_size,
            random_state=random_state,
            stratify=stratify_target,
        )
        return X_train, X_test, y_train, y_test


### How to Use It

| # | Method                   | Description                              |
|---|--------------------------|------------------------------------------|
| 1 | `get_clean_data()`       | Full cleaned dataframe                   |
| 2 | `get_features_target()`  | Modeling features and target             |
| 3 | `get_train_test_split()` | Reproducible encoded train/test split    |

In [ ]:
df_diabetic_data = pd.read_csv(os.path.join('..', 'data', 'diabetic_data.csv'))

In [ ]:
df_diabetic_loader = DiabeticDataLoader(df_diabetic_data)
df_diabetic_clean_loader = df_diabetic_loader.get_clean_data()
df_diabetic_features, df_diabetic_target = df_diabetic_loader.get_features_target()
df_X_train, df_X_test, df_y_train, df_y_test = df_diabetic_loader.get_train_test_split()